![PwC Logo](pwc-logo.png)

#🌐 Telecom Customer Churn Analysis: A PwC Case Study


## 📌 Background

**Who this is for.** The retention lead in the consumer division, with the CFO as secondary
audience — the first needs a list of accounts to act on, the second needs the revenue at stake
quantified before funding retention offers.

Telecom is a crowded market, and customers don't hesitate to switch providers when they find a
better deal, better service, or simply a better experience. Each customer who leaves takes their
monthly revenue with them, and those losses quietly add up across a base this size. So before any
fix can be recommended, a few real questions need answers:

- **Who is actually churning?** Which segments sit above the base rate, and by how much.
- **What's pushing them out the door?** Which attributes and service experiences travel with churn.
- **And how early can the business spot it coming?** Whether any observable behaviour separates
  leavers from stayers in time to intervene.

**In scope.** Descriptive analysis of one customer-level snapshot: segment churn rates, the
recurring revenue attached to churners, service and support patterns, high-value churner profiling,
and a transparent additive risk score usable without a modelling pipeline.

**Out of scope.** Causal attribution (the drivers overlap heavily and are not separated here),
predictive modelling with holdout validation, customer lifetime value or margin analysis (no cost
data), offer design and pricing, competitor and market context, and any time-series or cohort
view — the file has no dates.

**Assumptions carried through the analysis.**

1. `Churn = Yes` is a completed departure at snapshot time, and the recorded attributes describe
   the customer *before* leaving. Nothing in the file proves this ordering.
2. Ticket counts cover a recent window rather than the full customer lifetime. Undocumented — see
   the limitations note in the README.
3. `MonthlyCharges` is a stable proxy for recurring revenue, so summing it over a segment gives
   that segment's MRR. Ignores discounts ending, downgrades and one-off charges.

This notebook works through the data to answer those questions, building toward a practical way to
flag at-risk customers before they leave.

## 🗺️ Analysis Roadmap

**Objective:** Walk through the data in stages, from importing and cleaning to spotting churn drivers and scoring risk.

**1. Import the Data**
- What does the raw dataset bring in, and how many customers and features are we actually working with?

**2. Data Cleaning**
- Are there formatting issues or missing values that need fixing before the real analysis can begin?

**3. Churn Patterns**
- Which customers are churning, and how does that shift across contract type, tenure, and the services they use?

**4. Revenue at Risk**
- How much recurring revenue is tied up in customers who are likely to leave?

**5. Early Warning Signals**
- What patterns in service bundles or support activity show up before a customer actually churns?

**6. Risk Scoring**
- With the drivers understood, how can the business flag at-risk customers going forward?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

# PwC brand color palette
PWC_COLORS = {
    'orange': '#E0301E',
    'amber': '#EB8C00',
    'black': '#2D2D2D',
    'gray': '#8C8C8C',
    'light_gray': '#DEDEDE',
}
PWC_PALETTE = [PWC_COLORS['orange'], PWC_COLORS['black'], PWC_COLORS['amber'], PWC_COLORS['gray']]
PWC_CMAP = LinearSegmentedColormap.from_list('pwc_diverging', [PWC_COLORS['black'], '#FFFFFF', PWC_COLORS['orange']])

sns.set_theme(style='whitegrid', palette=PWC_PALETTE)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=PWC_PALETTE)

df = pd.read_excel('02 Churn-Dataset.xlsx')
print(df.shape)

def churn_bar(ax, x, rates, title, xlabel, baseline=None, highlight_from=None, counts=None):
    """Churn-rate bar chart in PwC styling. Replaces the repeated plotting block.

    rates          proportions (0-1), not percentages
    baseline       draws the overall churn rate as a dashed reference line
    highlight_from bars with x >= this value are orange, the rest grey
    counts         optional n per bar, printed under each tick label
    """
    if highlight_from is None:
        colors = PWC_COLORS['orange']
    else:
        colors = [PWC_COLORS['orange'] if v >= highlight_from else PWC_COLORS['gray'] for v in x]

    bars = ax.bar(x, rates, color=colors)

    if baseline is not None:
        ax.axhline(baseline, color=PWC_COLORS['black'], linestyle='--', linewidth=1.2,
                   label=f'Base rate {baseline:.0%}')
        ax.legend(frameon=False, loc='upper left', fontsize=8)

    for bar, rate in zip(bars, rates):
        ax.annotate(f'{rate:.0%}', (bar.get_x() + bar.get_width() / 2, rate),
                    ha='center', va='bottom', fontsize=8, color=PWC_COLORS['black'])

    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title(title, fontsize=11, loc='left')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Churn rate')
    ax.margins(y=0.18)
    if counts is not None:
        ax.set_xticks(list(x))
        ax.set_xticklabels([f'{v}\nn={n:,}' for v, n in zip(x, counts)], fontsize=8)
    return ax

## 1. Data Cleaning

Every decision below is recorded with its reason and its effect on the row count. No rows are
dropped at any stage: 7,043 in, 7,043 out.

| Step | Rows before → after | Decision and reason |
|---|---|---|
| Load | — → 7,043 | 23 columns, `customerID` unique, no duplicate rows |
| `TotalCharges` type | 7,043 → 7,043 | stored as text with blanks; coerced to numeric, producing 11 NaN |
| `TotalCharges` blanks | 7,043 → 7,043 | all 11 are `tenure == 0`, i.e. billed nothing yet. Filled with 0 rather than dropped — these are genuine new customers, and dropping them would bias the shortest-tenure bucket, which has the highest churn |
| Target encoding | 7,043 → 7,043 | `Churned = (Churn == 'Yes')` |
| Outliers | 7,043 → 7,043 | none removed; `tenure` 0–72, `MonthlyCharges` 18.25–118.75, `TotalCharges` 0–8,684.80 all sit within plausible product ranges |
| Structural nulls | — | "No internet service" / "No phone service" kept as their own level rather than recoded to "No" — they mark non-applicability, not a customer choice (see section 10) |

In [ ]:
# TotalCharges is object — coerce whitespace/empty to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Customers with tenure=0 have blank TotalCharges — fill with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Binary encode target
df['Churned'] = (df['Churn'] == 'Yes').astype(int)

print('Missing values after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nChurn rate: {df["Churned"].mean():.1%}')

# Verify the cleaning claims rather than asserting them
print(f'\nRows: {len(df):,}  |  unique customerIDs: {df["customerID"].nunique():,}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Zero-tenure customers: {(df["tenure"] == 0).sum()}  '
      f'|  all now have TotalCharges 0: {(df.loc[df["tenure"] == 0, "TotalCharges"] == 0).all()}')
print(f'Churned customers: {df["Churned"].sum():,}')

## 2. Overview

In [ ]:
df.describe(include='all').T

## 3. Churn Distribution

In [ ]:
counts = df['Churn'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=['#8C8C8C', '#E0301E'], width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
            f'{bar.get_height():,}', ha='center', va='bottom', fontsize=11)
ax.set_title('Churn vs Retained Customers', fontsize=13, pad=10)
ax.set_ylabel('Customers')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()
print(counts.to_string())

## 4. Churn by Categorical Features

In [ ]:
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rates = df.groupby(col)['Churned'].mean().sort_values(ascending=False)
    axes[i].bar(churn_rates.index.astype(str), churn_rates.values * 100,
                color=sns.color_palette(PWC_PALETTE, len(churn_rates)))
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Churn %')
    axes[i].set_ylim(0, 100)
    axes[i].tick_params(axis='x', rotation=25, labelsize=8)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Churn Rate by Categorical Feature', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Churn by Numerical Features

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'numAdminTickets', 'numTechTickets']

fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))

for ax, col in zip(axes, num_cols):
    for label, color in [('No', '#8C8C8C'), ('Yes', '#E0301E')]:
        subset = df[df['Churn'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel(col)
    ax.legend(title='Churned')

plt.suptitle('Distribution of Numerical Features by Churn', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap (Numerical)

In [ ]:
num_df = df[num_cols + ['Churned']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap=PWC_CMAP, center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Tenure vs Monthly Charges

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, color, marker in [('No', '#8C8C8C', 'o'), ('Yes', '#E0301E', 'x')]:
    subset = df[df['Churn'] == label]
    ax.scatter(subset['tenure'], subset['MonthlyCharges'],
               alpha=0.3, s=15, color=color, marker=marker, label=label)
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Monthly Charges ($)')
ax.set_title('Tenure vs Monthly Charges by Churn', fontsize=13)
ax.legend(title='Churned')
plt.tight_layout()
plt.show()

## 8. Key Takeaways

In [ ]:
overall_churn = df['Churned'].mean()

print(f'Overall churn rate: {overall_churn:.1%}\n')
print('--- Highest churn segments ---')

segments = {
    'Contract: Month-to-month': df[df['Contract'] == 'Month-to-month']['Churned'].mean(),
    'Internet: Fiber optic': df[df['InternetService'] == 'Fiber optic']['Churned'].mean(),
    'No Online Security': df[df['OnlineSecurity'] == 'No']['Churned'].mean(),
    'No Tech Support': df[df['TechSupport'] == 'No']['Churned'].mean(),
    'Paperless Billing: Yes': df[df['PaperlessBilling'] == 'Yes']['Churned'].mean(),
    'Senior Citizen': df[df['SeniorCitizen'] == 1]['Churned'].mean(),
    'Tenure < 12 months': df[df['tenure'] < 12]['Churned'].mean(),
}

for segment, rate in sorted(segments.items(), key=lambda x: -x[1]):
    print(f'  {segment}: {rate:.1%}  ({rate/overall_churn:.1f}x baseline)')

## 9. Revenue at Risk
Who churns matters less than *how much revenue* they take with them.

In [ ]:
total_mrr = df['MonthlyCharges'].sum()
churner_mrr = df[df['Churned'] == 1]['MonthlyCharges'].sum()
churner_pct_customers = df['Churned'].mean()
churner_pct_revenue = churner_mrr / total_mrr

print(f'Total MRR:          ${total_mrr:,.0f}')
print(f'MRR lost to churn:  ${churner_mrr:,.0f}  ({churner_pct_revenue:.1%} of revenue)')
print(f'Churners are {churner_pct_customers:.1%} of customers but {churner_pct_revenue:.1%} of revenue')
print()

rev_by_contract = df.groupby('Contract').agg(
    customers=('customerID', 'count'),
    churners=('Churned', 'sum'),
    churn_rate=('Churned', 'mean'),
    mrr_at_risk=('MonthlyCharges', lambda x: x[df.loc[x.index, 'Churned'] == 1].sum())
).reset_index()
rev_by_contract['mrr_at_risk_pct'] = rev_by_contract['mrr_at_risk'] / total_mrr
print(rev_by_contract.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

retained_mrr = total_mrr - churner_mrr
axes[0].pie([retained_mrr, churner_mrr],
            labels=['Retained', 'At Risk (Churners)'],
            colors=['#8C8C8C', '#E0301E'],
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Monthly Revenue: Retained vs At Risk', fontsize=12)

axes[1].bar(rev_by_contract['Contract'], rev_by_contract['mrr_at_risk'],
            color=['#E0301E', '#EB8C00', '#8C8C8C'])
axes[1].set_title('MRR Lost to Churn by Contract Type', fontsize=12)
axes[1].set_ylabel('Monthly Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, (_, row) in enumerate(rev_by_contract.iterrows()):
    axes[1].text(i, row['mrr_at_risk'] + 500, f"{row['churn_rate']:.0%} churn",
                 ha='center', fontsize=9, color='#2D2D2D')

plt.tight_layout()
plt.show()

## 10. Service Bundle Effect
Does subscribing to more services make customers stickier?

Add-ons only exist for customers who buy internet, so "No internet service" is a not-applicable
marker rather than a zero on the same scale. The two populations are counted separately below.

In [ ]:
ADDONS = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
          'TechSupport', 'StreamingTV', 'StreamingMovies']

# Counting "No internet service" as zero mixes two opposite populations: phone-only customers
# (low churn, minimal relationship) and internet customers who bought no add-ons (very high churn).
# Pooling them produces a non-monotonic curve, so they are reported separately.
df['AddonCount'] = df[ADDONS].eq('Yes').sum(axis=1)
has_internet = df['InternetService'].ne('No')

bundle = (df[has_internet]
          .groupby('AddonCount')
          .agg(customers=('customerID', 'count'),
               churn_rate=('Churned', 'mean'),
               avg_monthly=('MonthlyCharges', 'mean'))
          .reset_index())
print('Internet customers only:')
print(bundle.to_string(index=False))

phone_only = df[~has_internet]
print(f'\nPhone-only customers: {len(phone_only):,}  '
      f'|  churn {phone_only["Churned"].mean():.1%}  '
      f'|  avg monthly ${phone_only["MonthlyCharges"].mean():.2f}')

fig, ax = plt.subplots(figsize=(9, 4))
churn_bar(ax, bundle['AddonCount'], bundle['churn_rate'],
          'Each internet add-on lowers churn: 52% with none, 5% with all six',
          'Internet add-ons subscribed (of 6)',
          baseline=df['Churned'].mean(),
          counts=bundle['customers'])
plt.tight_layout()
plt.show()

## 11. The Fiber Optic Paradox
Fiber customers pay more *and* churn more. What's actually driving it?

In [ ]:
fiber = df[df['InternetService'] == 'Fiber optic'].copy()
dsl = df[df['InternetService'] == 'DSL'].copy()

print('=== Fiber Optic vs DSL ===')
for label, subset in [('Fiber optic', fiber), ('DSL', dsl)]:
    print(f"\n{label}  (n={len(subset):,})")
    print(f"  Churn rate:       {subset['Churned'].mean():.1%}")
    print(f"  Avg monthly:      ${subset['MonthlyCharges'].mean():.2f}")
    print(f"  Avg tech tickets: {subset['numTechTickets'].mean():.2f}")
    print(f"  Month-to-month:   {(subset['Contract'] == 'Month-to-month').mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cross = df[df['InternetService'].isin(['Fiber optic', 'DSL'])].groupby(
    ['InternetService', 'Contract'])['Churned'].mean().unstack()
cross.T.plot(kind='bar', ax=axes[0], color=['#E0301E', '#8C8C8C'], width=0.6)
axes[0].set_title('Churn Rate: Fiber vs DSL by Contract', fontsize=11)
axes[0].set_ylabel('Churn Rate')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].legend(title='Internet')

for label, color in [('Retained', '#8C8C8C'), ('Churned', '#E0301E')]:
    subset = fiber[fiber['Churned'] == (1 if label == 'Churned' else 0)]
    axes[1].hist(subset['numTechTickets'], bins=range(0, 10),
                 alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Tech Tickets: Fiber Churners vs Retained', fontsize=11)
axes[1].set_xlabel('Number of Tech Support Tickets')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Support Tickets as a Churn Signal
Is there a ticket volume tipping point where churn spikes?

Admin and tech tickets are examined separately rather than summed: pooling them hides that only one
of the two carries any churn signal.

In [ ]:
# Admin and tech tickets are NOT the same signal, so they are not added together.
for col in ['numAdminTickets', 'numTechTickets']:
    grp = (df.groupby(col)
             .agg(customers=('customerID', 'count'), churn_rate=('Churned', 'mean'))
             .reset_index())
    print(f'--- {col} ---')
    print(grp.to_string(index=False), '\n')

base = df['Churned'].mean()
print(f'Base rate:       {base:.1%}')
print(f'Any admin ticket (n={(df["numAdminTickets"] >= 1).sum():,}): '
      f'{df.loc[df["numAdminTickets"] >= 1, "Churned"].mean():.1%}  -> no signal')
print(f'Any tech ticket  (n={(df["numTechTickets"] >= 1).sum():,}): '
      f'{df.loc[df["numTechTickets"] >= 1, "Churned"].mean():.1%}  -> 2.6x base')

# Band the sparse tail so no bar rests on fewer than 20 customers
df['TechTicketBand'] = df['numTechTickets'].clip(upper=6)
admin = (df.groupby('numAdminTickets')
           .agg(customers=('customerID', 'count'), churn_rate=('Churned', 'mean')).reset_index())
tech = (df.groupby('TechTicketBand')
          .agg(customers=('customerID', 'count'), churn_rate=('Churned', 'mean')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
churn_bar(axes[0], admin['numAdminTickets'], admin['churn_rate'],
          'Admin tickets track nothing: churn is flat at every volume',
          'Admin tickets', baseline=base, counts=admin['customers'])
churn_bar(axes[1], tech['TechTicketBand'], tech['churn_rate'],
          'The first tech ticket triples churn risk (20% -> 66%)',
          'Tech tickets (6 = six or more)', baseline=base,
          highlight_from=1, counts=tech['customers'])
plt.tight_layout()
plt.show()

# Does the tech-ticket signal survive controlling for tenure?
tenure_band = pd.cut(df['tenure'], [-1, 11, 24, 48, 72],
                     labels=['0-11m', '12-24m', '25-48m', '49-72m'])
controlled = (pd.crosstab(tenure_band, df['numTechTickets'] >= 1, df['Churned'], aggfunc='mean')
                .rename(columns={False: 'no tech ticket', True: 'any tech ticket'}))
print('\nChurn rate by tenure band x any tech ticket:')
print(controlled.map(lambda v: f'{v:.1%}').to_string())

## 13. High-Value Churner Profile
The customers who hurt most to lose — high paying, already gone.

In [ ]:
churners = df[df['Churned'] == 1].copy()

# tenure * MonthlyCharges is revenue already collected, i.e. historical. Naming it CLV would be
# wrong: customer lifetime value is forward-looking and needs expected remaining lifetime and
# margin, neither of which exists in this file.
df['HistoricalRevenue'] = df['tenure'] * df['MonthlyCharges']
churners['HistoricalRevenue'] = churners['tenure'] * churners['MonthlyCharges']

charge_threshold = df['MonthlyCharges'].quantile(0.80)
high_value_churners = churners[churners['MonthlyCharges'] >= charge_threshold]

print(f'Charge threshold (top 20%): ${charge_threshold:.2f}/month')
print(f'High-value churners: {len(high_value_churners):,}  ({len(high_value_churners)/len(churners):.1%} of all churners)')
print(f'Revenue at risk from high-value churners: ${high_value_churners["MonthlyCharges"].sum():,.0f}/month')
print()
print('High-value churner profile:')
print(f'  Avg monthly charge: ${high_value_churners["MonthlyCharges"].mean():.2f}')
print(f'  Avg tenure:         {high_value_churners["tenure"].mean():.1f} months')
print(f'  Month-to-month:     {(high_value_churners["Contract"] == "Month-to-month").mean():.1%}')
print(f'  Fiber optic:        {(high_value_churners["InternetService"] == "Fiber optic").mean():.1%}')
print(f'  No tech support:    {(high_value_churners["TechSupport"] == "No").mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

retained = df[df['Churned'] == 0]
low_churners = churners[churners['MonthlyCharges'] < charge_threshold]

axes[0].scatter(retained['tenure'], retained['MonthlyCharges'],
                alpha=0.2, s=10, color='#DEDEDE', label='Retained')
axes[0].scatter(low_churners['tenure'], low_churners['MonthlyCharges'],
                alpha=0.4, s=15, color='#EB8C00', label='Churned (low value)')
axes[0].scatter(high_value_churners['tenure'], high_value_churners['MonthlyCharges'],
                alpha=0.7, s=25, color='#E0301E', label='Churned (high value)')
axes[0].axhline(charge_threshold, color='#E0301E', linestyle='--', linewidth=1,
                alpha=0.6, label=f'Top 20% threshold (${charge_threshold:.0f})')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Monthly Charges ($)')
axes[0].set_title('High-Value Churners in Context', fontsize=11)
axes[0].legend(markerscale=1.5, fontsize=8)

axes[1].hist(df[df['Churned'] == 0]['HistoricalRevenue'], bins=40, alpha=0.6, color='#8C8C8C',
             label='Retained', density=True)
axes[1].hist(churners['HistoricalRevenue'], bins=40, alpha=0.6, color='#E0301E',
             label='Churned', density=True)
axes[1].set_xlabel('Revenue collected to date (tenure x monthly charge)')
axes[1].set_ylabel('Density')
axes[1].set_title('Revenue collected to date: retained vs churned', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. Customer Risk Scoring
Flag active customers most likely to leave using additive risk signals — no ML needed.

Six equally weighted flags. `SeniorCitizen` is deliberately excluded: age is a protected
characteristic and unsuitable for selecting customers for offers, and dropping it improved the
score's separation. The ticket flag uses tech tickets only, at one or more, following section 12.
The score is validated in-sample, so the gradient below is fitted, not predictive performance.

In [ ]:
# Six transparent flags. SeniorCitizen excluded on fairness grounds (and it improved separation).
# Ticket flag uses tech tickets only, since admin tickets are flat against churn (section 12).
df['risk_mtm']     = (df['Contract'] == 'Month-to-month').astype(int)
df['risk_fiber']   = (df['InternetService'] == 'Fiber optic').astype(int)
df['risk_no_sec']  = (df['OnlineSecurity'] == 'No').astype(int)
df['risk_no_sup']  = (df['TechSupport'] == 'No').astype(int)
df['risk_new']     = (df['tenure'] < 12).astype(int)
df['risk_techtkt'] = (df['numTechTickets'] >= 1).astype(int)

RISK_FLAGS = ['risk_mtm', 'risk_fiber', 'risk_no_sec', 'risk_no_sup', 'risk_new', 'risk_techtkt']
df['RiskScore'] = df[RISK_FLAGS].sum(axis=1)

validation = (df.groupby('RiskScore')
                .agg(customers=('customerID', 'count'), churn_rate=('Churned', 'mean'))
                .reset_index())
validation['lift_vs_base'] = validation['churn_rate'] / df['Churned'].mean()
print('Score vs actual churn (IN-SAMPLE: flags were chosen from this same data)')
print(validation.to_string(index=False))

print('\nOperating points across the whole base:')
for threshold in (3, 4, 5):
    flagged = df[df['RiskScore'] >= threshold]
    print(f'  score >= {threshold}: {len(flagged):,} customers '
          f'({len(flagged) / len(df):.0%} of base), '
          f'{flagged["Churned"].mean():.1%} of them churned, '
          f'capturing {flagged["Churned"].sum() / df["Churned"].sum():.0%} of all churners')

active = df[df['Churned'] == 0]
print()
for threshold in (4, 5):
    target = active[active['RiskScore'] >= threshold]
    print(f'Active customers at score >= {threshold}: {len(target):,}  '
          f'|  MRR ${target["MonthlyCharges"].sum():,.0f}/month')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

churn_bar(axes[0], validation['RiskScore'], validation['churn_rate'],
          'Risk score tracks churn: 1% at zero flags, 77% at five',
          'Risk score (0-6)', baseline=df['Churned'].mean(),
          highlight_from=4, counts=validation['customers'])

active_risk = active['RiskScore'].value_counts().sort_index()
bar_colors = [PWC_COLORS['orange'] if r >= 4 else PWC_COLORS['gray'] for r in active_risk.index]
axes[1].bar(active_risk.index, active_risk.values, color=bar_colors)
axes[1].set_xlabel('Risk score (0-6)')
axes[1].set_ylabel('Active customers')
axes[1].set_title('Where the still-active base sits: the orange bars are the outreach list',
                  fontsize=11, loc='left')
for x_pos, count in zip(active_risk.index, active_risk.values):
    axes[1].annotate(f'{count:,}', (x_pos, count), ha='center', va='bottom', fontsize=8,
                     color=PWC_COLORS['black'])
axes[1].margins(y=0.15)

plt.tight_layout()
plt.show()